# The sampler: Algorithm 3, one trajectory at a time

`SpeculativeSampler` owns the loop and the bookkeeping and **nothing else** — topology lives in
`trees.py`, the models in `kernels.py`, the coupling in `verify.py`. It is a transcription of
the paper's pseudocode.

A round starting at step `n` has three phases:

| phase | lines | what happens |
| --- | --- | --- |
| **1. draft** | 5–9 | expand the tree level by level with `m^p`; sequential in depth, parallel within a level |
| **2. verify** | 11–13 | **one** batched target call, over the tree's *internal* nodes only |
| **3. accept** | 15–23 | walk down from the root, stopping at the first rejection |

The round commits between `1` and `L_n` states for that single target call, and
`L_n = min(L, N - n)` truncates the tree near the horizon (eq. 27). That ratio is the speedup.

In [1]:
import numpy as np

from specdiff import (
    DelayedDriftProposal,
    DraftTree,
    NoiseSchedule,
    ProposalTransition,
    SpeculativeSampler,
    TargetTransition,
    Verifier,
    VerifyResult,
    standard_sampler,
)
from specdiff.ops import standard_normal_sf
from specdiff.verifiers.rank1 import Rank1Frame

rng = np.random.default_rng(0)

## 1. A toy model

The same linear reverse kernel as the kernels tutorial: `m^q_n(y) = y + gamma_n (c - y)` with
`c = 0`. It is linear, so the exact law of `Y_N` is available in closed form and every claim
below can be checked rather than believed.

In [2]:
dim, N = 4, 20


class LinearReverseKernel(TargetTransition):
    """Stand-in for the expensive network: m^q_n(y) = y + gamma_n (c - y)."""

    def __init__(self, gammas):
        super().__init__()
        self.gammas = np.asarray(gammas, dtype=float)

    def means(self, states, steps):
        gamma = self.gammas[list(steps)].reshape(-1, *([1] * (states.ndim - 1)))
        return states * (1.0 - gamma)


class SqrtSchedule(NoiseSchedule):
    def __init__(self, num_steps, scale=0.4, floor=0.05):
        self.num_steps, self.scale, self.floor = int(num_steps), float(scale), float(floor)

    def sigma(self, step):
        return self.floor + self.scale * np.sqrt((self.num_steps - step) / self.num_steps)


target = LinearReverseKernel(np.linspace(0.05, 0.25, N))
schedule = SqrtSchedule(N)


def exact_law(y0):
    """Closed-form N(mean, var I) of Y_N for this linear kernel, for checking."""
    mean, var = np.asarray(y0, dtype=float), 0.0
    for n in range(N):
        a, s = 1.0 - target.gammas[n], schedule(n)
        mean, var = a * mean, a * a * var + s * s
    return mean, var


y0 = np.zeros(dim)
mean_N, var_N = exact_law(y0)
print("exact law of Y_N:  mean", mean_N.round(4), " std", round(float(np.sqrt(var_N)), 4))

exact law of Y_N:  mean [0. 0. 0. 0.]  std 0.3006


## 2. The reference: `standard_sampler`

`standard_sampler` is Algorithm 3 in its degenerate case — `K = L = 1` with a rule that always
resamples. One committed state per target call, i.e. `N` NFEs and `1.00x`. It is the
denominator of every speedup number and the distributional ground truth.

In [3]:
reference = standard_sampler(target, schedule, num_steps=N)
result = reference.sample(y0, rng=rng)

print(result.summary())
print()
print("trajectory shape        ", result.trajectory.shape, " (Y_0 ... Y_N)")
print("sample (= trajectory[-1])", result.sample.round(3))
print("rounds                  ", len(result.rounds))
print("target_calls (NFEs)     ", result.target_calls)
print("target_states_evaluated ", result.target_states_evaluated)
print("drafted_states          ", result.drafted_states)
print("speedup                 ", result.speedup)
print("acceptance_rate         ", result.acceptance_rate)

steps=20 target_calls=20 speedup=1.000x acceptance=0.000 drafted=20 verified=20

trajectory shape         (21, 4)  (Y_0 ... Y_N)
sample (= trajectory[-1]) [-0.169  0.337 -0.194  0.157]
rounds                   20
target_calls (NFEs)      20
target_states_evaluated  20
drafted_states           20
speedup                  1.0
acceptance_rate          0.0


In [4]:
# It reproduces the closed-form law -- 2000 independent runs.
def terminal_samples(sampler, trials, seed=1):
    r = np.random.default_rng(seed)
    return np.stack([sampler.sample(y0, rng=r, record=False).sample for _ in range(trials)])


ref_samples = terminal_samples(reference, 2000)
print(f"empirical mean {ref_samples.mean(axis=0).round(3)}   std {ref_samples.std(axis=0).round(3)}")
print(f"exact     mean {mean_N.round(3)}   std {np.sqrt(var_N):.3f}")

empirical mean [ 0.002 -0.01  -0.004 -0.003]   std [0.305 0.304 0.306 0.302]
exact     mean [0. 0. 0. 0.]   std 0.301


## 3. A speculative run

Swap in a draft tree, the delayed-drift proposal, and a rule. The rule here is the delta probe
from the verifier tutorial: exact, but it never accepts. That is deliberate — it isolates the
*sampler*, and it measures the mismatch `delta` that any future coupling will have to work
against.

`check_contract=True` wraps the rule in `CheckedVerifier`. It is cheap; leave it on while
developing.

In [5]:
class DeltaProbe(Verifier):
    """Records the mean mismatch at every node, then resamples exactly."""

    name = "delta-probe"

    def __init__(self):
        self.deltas = []

    def reset(self):
        self.deltas.clear()

    def verify(self, request):
        frame = Rank1Frame.from_request(request)
        self.deltas.append(frame.delta)
        ops = self.backend_for(request)
        noise = ops.randn_stack(1, request.target_mean, request.rng)[0]
        return VerifyResult(request.target_mean + request.sigma * noise, accepted=False)


tree = DraftTree.uniform(branching=2, lookahead=3)
probe = DeltaProbe()

sampler = SpeculativeSampler(
    target=target,
    proposal=DelayedDriftProposal(target),     # eq. (7), free per round via prefetching
    schedule=schedule,
    tree=tree,
    verifier=probe,
    num_steps=N,
    check_contract=True,
)

spec = sampler.sample(y0, rng=rng)
print(tree)
print(spec.summary())

DraftTree(uniform, K=2, L=3, B=14, |I|=7)
steps=20 target_calls=21 speedup=0.952x acceptance=0.000 drafted=260 verified=131


In [6]:
# Same law as the reference: the sampler plus an exact rule preserves the target chain.
spec_samples = terminal_samples(sampler, 2000)
print(f"speculative  mean {spec_samples.mean(axis=0).round(3)}   std {spec_samples.std(axis=0).round(3)}")
print(f"reference    mean {ref_samples.mean(axis=0).round(3)}   std {ref_samples.std(axis=0).round(3)}")
print(f"exact        mean {mean_N.round(3)}   std {np.sqrt(var_N):.3f}")

speculative  mean [-0.004  0.009 -0.004  0.001]   std [0.298 0.301 0.306 0.299]
reference    mean [ 0.002 -0.01  -0.004 -0.003]   std [0.305 0.304 0.306 0.302]
exact        mean [0. 0. 0. 0.]   std 0.301


`speedup = 1.00x` and `acceptance = 0.000` is exactly right for a probe: it rejects at every
node, so a round commits one state and costs one target call — the same as the reference, but
with `B` wasted drafts. What the run bought is the measurement:

In [7]:
deltas = np.array(probe.deltas)
alpha = 2.0 * standard_normal_sf(deltas.mean() / 2.0)     # eq. (16): K = 1 acceptance
print(f"nodes verified       {len(deltas)}")
print(f"mean delta           {deltas.mean():.4f}   (median {np.median(deltas):.4f})")
print(f"implied K=1 accept   {alpha:.3f}       [eq. 16]")
print(f"implied chain ceiling {1.0 / (1.0 - alpha):.2f}x     [Appendix D.1]")
print()
print("Caveat: a probe that never accepts advances one step per round, so the delayed drift")
print("is never more than one step stale. This delta is a *lower* bound; under a real coupling")
print("the drift ages across the accepted prefix and the mismatch grows.")

nodes verified       20
mean delta           0.3159   (median 0.2731)
implied K=1 accept   0.875       [eq. 16]
implied chain ceiling 7.97x     [Appendix D.1]

Caveat: a probe that never accepts advances one step per round, so the delayed drift
is never more than one step stale. This delta is a *lower* bound; under a real coupling
the drift ages across the accepted prefix and the mismatch grows.


## 4. Where the cost goes

Three different numbers, and they mean different things:

| number | meaning |
| --- | --- |
| `target_calls` | **NFEs** — one batched call per round, plus a proposal's warm-up. The paper's cost metric. |
| `target_states_evaluated` | batch *volume*: total rows pushed through the target, `|I|` per round |
| `drafted_states` | states the proposal produced, `B` per round |

The target is evaluated only at *internal* nodes — leaves are never parents, so their target
means are never needed. That is `|I| = B / K` for a uniform tree, and it is where a tree buys
back part of its verification cost.

In [8]:
print(f"{'topology':>22}{'K':>4}{'L':>4}{'B':>6}{'|I|':>6}{'NFEs':>7}{'rows':>8}{'drafts':>8}")
for label, t in (
    ("chain(3)", DraftTree.chain(3)),
    ("uniform(2, 3)", DraftTree.uniform(branching=2, lookahead=3)),
    ("uniform(4, 3)", DraftTree.uniform(branching=4, lookahead=3)),
    ("from_widths([4,2,1])", DraftTree.from_widths([4, 2, 1])),     # depth-dependent widths
):
    s = SpeculativeSampler(
        target=target, proposal=DelayedDriftProposal(target), schedule=schedule,
        tree=t, verifier=DeltaProbe(), num_steps=N,
    )
    r = s.sample(y0, rng=np.random.default_rng(3))
    print(f"{label:>22}{t.branching:>4}{t.depth:>4}{t.budget:>6}{len(t.internal_nodes):>6}"
          f"{r.target_calls:>7}{r.target_states_evaluated:>8}{r.drafted_states:>8}")

              topology   K   L     B   |I|   NFEs    rows  drafts
              chain(3)   1   3     3     3     21      58      57
         uniform(2, 3)   2   3    14     7     21     131     260
         uniform(4, 3)   4   3    84    21     21     385    1536
  from_widths([4,2,1])   4   3    20    13     21     241     376


Note `from_widths([4, 2, 1])` — a non-uniform tree needs no special handling anywhere; `|I|` is
just "nodes with children". Note also that a wider tree costs *more* rows and *more* drafts per
round for the same `L`: the tree is a bet that more candidates per node raise the acceptance
probability enough to pay for it.

## 5. `RoundRecord`: what happened in each round

Every round returns a record. `committed` is the paper's `N_alpha` — states committed for that
one target call — and `lookahead` shows the truncation `T|_{L_n}` biting near the horizon.

In [9]:
run = sampler.sample(y0, rng=np.random.default_rng(7))

print(f"{'round':>6}{'start_step n':>14}{'L_n':>5}{'committed':>11}{'accepted_depth':>16}"
      f"{'rejected':>10}{'drafted B':>11}{'verified |I|':>14}")
for i, r in enumerate(run.rounds):
    if 3 <= i < len(run.rounds) - 4:
        continue                                  # the middle rounds all look alike
    if i == len(run.rounds) - 4:
        print(f"{'...':>6}")
    print(f"{i:>6}{r.start_step:>14}{r.lookahead:>5}{r.committed:>11}{r.accepted_depth:>16}"
          f"{str(r.rejected):>10}{r.drafted:>11}{r.verified:>14}")

 round  start_step n  L_n  committed  accepted_depth  rejected  drafted B  verified |I|
     0             0    3          1               0      True         14             7
     1             1    3          1               0      True         14             7
     2             2    3          1               0      True         14             7
   ...
    16            16    3          1               0      True         14             7
    17            17    3          1               0      True         14             7
    18            18    2          1               0      True          6             3
    19            19    1          1               0      True          2             1


The last rounds are the interesting ones: at `n = 18` there are only two steps left, so
`L_n = min(3, 20 - 18) = 2` and the tree is truncated to depth 2 — `B` drops from 14 to 6 and
`|I|` from 7 to 3. Nothing is rebuilt: node ids are in breadth-first order, so `T|_m` is a
slice (and it is cached per tree).

Rounds can also be streamed as they happen, which is how you drive a progress bar or an
adaptive policy without keeping every record:

In [10]:
seen = []
sampler.sample(
    y0,
    rng=np.random.default_rng(7),
    on_round=lambda rec: seen.append((rec.start_step, rec.committed)),
    record=False,                     # don't retain records in the result
)
print("first five (n, committed):", seen[:5])
print("records kept in result:   ", len(sampler.sample(y0, rng=rng, record=False).rounds))

first five (n, committed): [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1)]
records kept in result:    0


## 6. The acceptance path

A probe never accepts, so nothing above exercised phase 3's descent. To see it, make the
proposal *perfect*: `delta = 0` at every node, where Remark 2 says the acceptance probability is
1 for every `K` and a rule may accept unconditionally.

This is a **measurement device, not a model**. `OracleProposal` below cheats by evaluating the
target's own formula off the books; a real proposal is either a cheaper network or a stale
drift, and it costs you a nonzero `delta`. What the device shows is the sampler's ceiling: `L`
committed states per target call.

In [11]:
class OracleProposal(ProposalTransition):
    """m^p = m^q, computed off the books. delta = 0 everywhere."""

    def means(self, states, steps):
        return target.means(states, steps)      # .means, not target(...): uncounted


class AcceptIfIdentical(Verifier):
    """Accepts only in the regime where the two kernels coincide (Remark 2).

    Exact, and not a coupling: when the frame is degenerate P and Q are the same
    distribution at the state's own precision, so the first child is already a
    draw from the target. Otherwise it falls back to an exact resample.
    """

    name = "accept-if-identical"

    def verify(self, request):
        frame = Rank1Frame.from_request(request)
        if frame.degenerate:
            return VerifyResult(request.child(0), accepted=True, child_index=0)
        ops = self.backend_for(request)
        noise = ops.randn_stack(1, request.target_mean, request.rng)[0]
        return VerifyResult(request.target_mean + request.sigma * noise, accepted=False)


oracle = SpeculativeSampler(
    target=target, proposal=OracleProposal(), schedule=schedule,
    tree=DraftTree.uniform(branching=2, lookahead=3),
    verifier=AcceptIfIdentical(), num_steps=N, check_contract=True,
)
r = oracle.sample(y0, rng=np.random.default_rng(1))
print(r.summary())
print()
print(f"{'round':>6}{'n':>5}{'L_n':>5}{'committed':>11}{'accepted_depth':>16}{'rejected':>10}")
for i, rec in enumerate(r.rounds):
    print(f"{i:>6}{rec.start_step:>5}{rec.lookahead:>5}{rec.committed:>11}"
          f"{rec.accepted_depth:>16}{str(rec.rejected):>10}")

steps=20 target_calls=7 speedup=2.857x acceptance=1.000 drafted=90 verified=45

 round    n  L_n  committed  accepted_depth  rejected
     0    0    3          3               3     False
     1    3    3          3               3     False
     2    6    3          3               3     False
     3    9    3          3               3     False
     4   12    3          3               3     False
     5   15    3          3               3     False
     6   18    2          2               2     False


`L = 3` committed states per target call and `acceptance = 1.000`: 20 steps in 7 rounds, i.e.
`2.86x` — not quite `3x`, because the last round had only two steps left to commit. That is the
ceiling of this topology: a round can never commit more than `L_n` states, so **`L` bounds the
speedup and `alpha` decides how much of it you get**.

Swap the proposal back for the delayed drift and the same rule collapses to `1.00x`, because
`delta > 0` almost everywhere and this rule is not a coupling:

In [12]:
honest = SpeculativeSampler(
    target=target, proposal=DelayedDriftProposal(target), schedule=schedule,
    tree=DraftTree.uniform(branching=2, lookahead=3),
    verifier=AcceptIfIdentical(), num_steps=N,
)
print(honest.sample(y0, rng=np.random.default_rng(1)).summary())

steps=20 target_calls=20 speedup=1.000x acceptance=0.050 drafted=246 verified=124


The one acceptance in that run (`0.050 = 1/20`) is the very first round: the warm-up call made
the frozen drift exact at `Y_0`, so `delta = 0` there and the degenerate branch fired. From the
second round on, the prefetched drift belongs to the *previous* round's root and `delta > 0`.

Everything between those two numbers is what a real coupling is worth. `specdiff/verifiers/stubs.py`
is where Algorithms 1 and 2 go; the verifier tutorial has the recipe.

## 7. What a coupling would buy, before writing one

You do not need a rule to predict the payoff — only `alpha`, which the probe already measured.
Each round a chain accepts a `Geom(alpha)` prefix, capped at `L`, and then commits one more
state on the rejection, so it advances `min(Geom(alpha), L - 1) + 1` steps per target call. That
is arithmetic, not sampling:

In [13]:
def projected_speedup(alpha, lookahead, num_steps, trials=4000, seed=0):
    """Expected N / rounds for a chain-like topology at per-level acceptance alpha."""
    r = np.random.default_rng(seed)
    rounds = np.zeros(trials)
    steps = np.zeros(trials, dtype=int)
    while (steps < num_steps).any():
        live = steps < num_steps
        run = r.geometric(1.0 - alpha, size=trials) - 1              # accepted prefix
        advance = np.minimum(np.minimum(run, lookahead - 1) + 1, num_steps - steps)
        steps = np.where(live, steps + advance, steps)
        rounds += live
    return float(np.mean(num_steps / rounds))


measured_alpha = 2.0 * standard_normal_sf(np.mean(probe.deltas) / 2.0)
print(f"measured alpha on this model: {measured_alpha:.3f}")
print()
print(f"{'alpha':>7}" + "".join(f"{'L=' + str(l):>9}" for l in (1, 2, 3, 4, 8)))
for a in (0.3, 0.6, measured_alpha, 0.95):
    row = f"{a:>7.3f}"
    for l in (1, 2, 3, 4, 8):
        row += f"{projected_speedup(a, l, N):>8.2f}x"
    print(row)

measured alpha on this model: 0.886

  alpha      L=1      L=2      L=3      L=4      L=8
  0.300    1.00x    1.30x    1.39x    1.41x    1.43x
  0.600    1.00x    1.59x    1.94x    2.15x    2.44x
  0.886    1.00x    1.85x    2.57x    3.23x    4.98x
  0.950    1.00x    1.93x    2.74x    3.61x    5.93x


Two things to read off. `L = 1` is never worth anything (speculating one step ahead and then
committing it is the standard sampler), and the return on `L` saturates once `L` exceeds the
typical accepted run length `alpha / (1 - alpha)` — past that you are drafting states nobody will
ever use. The `alpha` column is where a tree earns its keep: more candidates per node raise
`alpha`, which is precisely what Algorithm 2 exists to exploit.

## 8. Guards

Misconfiguration fails at construction or on the first array, never silently.

In [14]:
# Integer states would truncate every Gaussian draw to zero and return a silently
# wrong trajectory, so sample() refuses one outright.
try:
    reference.sample(np.zeros(dim, dtype=int), rng=rng)
except TypeError as exc:
    print("TypeError ->", str(exc).splitlines()[0])

# A single-proposal rule against a branching tree: caught at construction.
class SingleProposalRule(DeltaProbe):
    max_children = 1

try:
    SpeculativeSampler(
        target=target, proposal=DelayedDriftProposal(target), schedule=schedule,
        tree=DraftTree.uniform(branching=3, lookahead=2),
        verifier=SingleProposalRule(), num_steps=N,
    )
except ValueError as exc:
    print("ValueError ->", exc)

# num_steps
try:
    SpeculativeSampler(target=target, proposal=DelayedDriftProposal(target), schedule=schedule,
                       tree=DraftTree.chain(2), verifier=DeltaProbe(), num_steps=0)
except ValueError as exc:
    print("ValueError ->", exc)

TypeError -> init has non-floating dtype dtype('int64'). Diffusion states must be floating point: an integer array truncates every Gaussian draw to zero and the sampler would return a silently wrong trajectory. Cast with e.g. `init.astype(float)`.
ValueError -> SingleProposalRule supports at most K=1 proposals per node, but the draft tree has K=3. Use DraftTree.chain(L) for single-proposal rules.
ValueError -> num_steps must be >= 1


In [15]:
# And with check_contract=True, a rule that breaks obligation 2 is caught in flight
# rather than corrupting the chain.
class BrokenRule(Verifier):
    name = "broken"

    def verify(self, request):
        return VerifyResult(request.child(0) + 1e-6, accepted=True, child_index=0)


bad = SpeculativeSampler(
    target=target, proposal=OracleProposal(), schedule=schedule, tree=DraftTree.chain(2),
    verifier=BrokenRule(), num_steps=N, check_contract=True,
)
try:
    bad.sample(y0, rng=rng)
except ValueError as exc:
    print("ValueError ->", str(exc)[:120], "...")

ValueError -> broken reported accepted=True but the returned state is not child 0. An accepted state must be the drafted state itself, ...


## 9. Reproducibility

Everything — drafting noise, and whatever the rule draws from `request.rng` — comes from the
generator passed to `sample`. Same seed, same trajectory.

In [16]:
a = sampler.sample(y0, rng=np.random.default_rng(42)).sample
b = sampler.sample(y0, rng=np.random.default_rng(42)).sample
c = sampler.sample(y0, rng=np.random.default_rng(43)).sample
print("same seed identical:", np.array_equal(a, b), "  different seed:", np.array_equal(a, c))

same seed identical: True   different seed: False


## Recap

* One round = draft (`B` states, `L` proposal calls) → verify (**1** target call over `|I|`
  nodes) → accept (descend until the first rejection).
* `committed` per round is the speedup; it is capped by `L_n = min(L, N - n)`.
* `target_calls` is the NFE metric; `target_states_evaluated` is batch volume; they are not the
  same thing and only the first is what the paper plots.
* An exact rule leaves the law of `Y_N` unchanged — checked above against the closed form.
* Measure `delta` with a probe first; `alpha = 2 Phi_bar(delta / 2)` and the arithmetic in
  section 7 tell you what a coupling is worth before you write one.

**Next:** [`batched_tutorial.ipynb`](batched_tutorial.ipynb) — the same algorithm over many
trajectories, where cost becomes a max rather than a mean.